# Colombia: price gaps and nutrition-aware review

This notebook recreates the Colombia evidence used in the Hoos' Cooking dashboard. It summarizes the directly comparable three-market basket and the pilot carrot–ahuyama price action.

> **Decision-support boundary:** these are official SIPSA wholesale market-price observations, not household retail prices or a subnational Cost of a Healthy Diet calculation. The lower-cost candidate is a prompt for human review, not an automatic substitution or dietary recommendation.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd() / 'colombia_sipsa_pilot', Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'data' / 'processed' / 'three_market_shared_basket_totals.csv').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from the repository or colombia_sipsa_pilot folder.')

ROOT = find_project_root()
PROCESSED = ROOT / 'data' / 'processed'
ROOT

PosixPath('/Users/ginamancuso/.codex/.chatgpt-projects/g-p-6a847acb9d1c8191be127194068698c8/publish_repo/colombia_sipsa_pilot')

## 1. Directly comparable shared basket

Only the eight foods observed in Bogotá, Medellín, and Montería on every sampled date are included. This protects the comparison from treating missing observations as prices.

In [2]:
totals = pd.read_csv(PROCESSED / 'three_market_shared_basket_totals.csv', parse_dates=['date'])
cities = {'Bogotá', 'Medellín', 'Montería'}
assert cities == set(totals['city']), 'Expected three Colombia markets.'
assert totals['foods_in_shared_basket'].eq(8).all(), 'The shared basket should contain eight foods.'

basket_by_date = (
    totals.pivot(index='date', columns='city', values='equal_weight_shared_basket_cop')
    .reindex(columns=['Bogotá', 'Medellín', 'Montería'])
    .round(0)
    .astype('int64')
)
basket_by_date

city,Bogotá,Medellín,Montería
date,,,
2025-02-12,18301,15297,14084
2025-05-14,18447,17461,15686
2025-08-13,22357,18555,17343
2025-11-12,18895,14717,15236


In [3]:
lowest_market = (
    totals.loc[totals.groupby('date')['equal_weight_shared_basket_cop'].idxmin(), ['date', 'city', 'equal_weight_shared_basket_cop']]
    .rename(columns={'city': 'lowest_observed_market', 'equal_weight_shared_basket_cop': 'shared_basket_cop'})
    .sort_values('date')
)
display(lowest_market)
print('Interpretation: the lowest observed market changes across sampled dates, which is why local and timely monitoring matters.')

,date,lowest_observed_market,shared_basket_cop
0,2025-02-12,Montería,14084.0
3,2025-05-14,Montería,15686.0
6,2025-08-13,Montería,17343.0
9,2025-11-12,Medellín,14717.0


Interpretation: the lowest observed market changes across sampled dates, which is why local and timely monitoring matters.


## 2. Nutrition-aware price action

The pilot examines one verified food role: orange vegetables. Ahuyama can appear as a potential lower-cost alternative only when it meets the documented vitamin A and fibre guardrails.

In [4]:
actions = pd.read_csv(PROCESSED / 'orange_vegetable_actions.csv', parse_dates=['date'])
bogota_actions = actions.loc[actions['market'].eq('Bogotá')].copy()
display(bogota_actions[[
    'date', 'higher_price_orange_vegetable', 'higher_price_cop_per_kg',
    'lower_cost_alternative', 'lower_cost_cop_per_kg',
    'potential_savings_cop_per_kg', 'action_status'
]])

august = bogota_actions.loc[bogota_actions['date'].eq(pd.Timestamp('2025-08-13'))].iloc[0]
scenario_kg = 500
gross_difference = august['potential_savings_cop_per_kg'] * scenario_kg
print(f"August Bogotá planning scenario: {scenario_kg:,} kg × COP {august['potential_savings_cop_per_kg']:,.0f}/kg = COP {gross_difference:,.0f}.")

,date,higher_price_orange_vegetable,higher_price_cop_per_kg,lower_cost_alternative,lower_cost_cop_per_kg,potential_savings_cop_per_kg,action_status
0,2025-02-12,Zanahoria,2177.0,Ahuyama,1675.0,502.0,Potential lower-cost alternative
2,2025-05-14,Ahuyama,1875.0,Zanahoria,1750.0,125.0,Potential lower-cost alternative
4,2025-08-13,Zanahoria,4271.0,Ahuyama,2913.0,1358.0,Potential lower-cost alternative
6,2025-11-12,Zanahoria,2000.0,Ahuyama,1900.0,100.0,Potential lower-cost alternative


August Bogotá planning scenario: 500 kg × COP 1,358/kg = COP 679,000.


In [5]:
nutrition = pd.read_csv(ROOT / 'data' / 'nutrition_lookup.csv')
comparison = nutrition.loc[nutrition['item'].isin(['Zanahoria', 'Ahuyama']), [
    'item', 'tcac_match_status', 'proposed_nutrition_role',
    'fiber_g_per_100g', 'vitamin_a_er_per_100g', 'iron_mg_per_100g',
    'calcium_mg_per_100g', 'vitamin_c_mg_per_100g'
]].set_index('item')
display(comparison)
print('Guardrails used in this pilot: vitamin A ≥ 1,000 ER and fibre ≥ 0.8 g per 100 g. Human review remains required.')

,tcac_match_status,proposed_nutrition_role,fiber_g_per_100g,vitamin_a_er_per_100g,iron_mg_per_100g,calcium_mg_per_100g,vitamin_c_mg_per_100g
item,,,,,,,
Zanahoria,verified_tcac_match,Orange vegetable role,0.8,1318.0,0.4,27,3.0
Ahuyama,verified_tcac_match,Orange vegetable role,1.1,1775.0,0.8,20,9.0


Guardrails used in this pilot: vitamin A ≥ 1,000 ER and fibre ≥ 0.8 g per 100 g. Human review remains required.


## Takeaway

The notebook supports a narrow, transparent result: price differences vary by food, market, and sampled date. The dashboard makes that result usable by pairing a price signal with an explicit nutrition comparison and review gate. It does not diagnose causes, predict shortages, or issue a procurement decision.